# 04 — Weak Supervision (Snorkel)

Six labeling functions (keyword + regex heuristics) vote on a label;
Snorkel's `LabelModel` combines them probabilistically. If Snorkel fails
to install (numpy/pandas version pinning issues are common on Windows),
see the commented Plan B cell at the bottom for a plain-pandas fallback.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import json
import re

import pandas as pd
from snorkel.labeling import LFAnalysis, PandasLFApplier, labeling_function
from snorkel.labeling.model import LabelModel

from utils import config
from utils.data import stratified_sample
from utils.metrics import evaluate_label_quality

ABSTAIN = -1
WORLD, SPORTS, BUSINESS, SCITECH = 0, 1, 2, 3  # must match utils.config.CLASS_NAMES order

In [2]:
WORLD_KEYWORDS = ["war", "government", "election", "military", "president",
                   "troops", "peace", "nato", "un ", "united nations"]
SPORTS_KEYWORDS = ["game", "player", "coach", "season", "championship",
                    "league", "tournament", "score", "athlete", "nfl", "nba"]
BUSINESS_KEYWORDS = ["stock", "market", "shares", "revenue", "profit",
                      "ceo", "earnings", "investor", "company", "merger"]
SCITECH_KEYWORDS = ["software", "technology", "research", "scientist",
                     "space", "nasa", "quantum", "ai", "computer", "drug"]


def keyword_lf(text, keywords, label):
    text_lower = text.lower()
    return label if any(kw in text_lower for kw in keywords) else ABSTAIN


@labeling_function()
def lf_world(x): return keyword_lf(x.text, WORLD_KEYWORDS, WORLD)


@labeling_function()
def lf_sports(x): return keyword_lf(x.text, SPORTS_KEYWORDS, SPORTS)


@labeling_function()
def lf_business(x): return keyword_lf(x.text, BUSINESS_KEYWORDS, BUSINESS)


@labeling_function()
def lf_scitech(x): return keyword_lf(x.text, SCITECH_KEYWORDS, SCITECH)


@labeling_function()
def lf_has_percentage(x):
    return BUSINESS if re.search(r'\d+\.?\d*\%', x.text) else ABSTAIN


@labeling_function()
def lf_has_score(x):
    return SPORTS if re.search(r'\b\d{1,2}-\d{1,2}\b', x.text) else ABSTAIN


lfs = [lf_world, lf_sports, lf_business, lf_scitech, lf_has_percentage, lf_has_score]

In [3]:
unlabeled_df = pd.read_parquet(config.PROCESSED_DIR / "unlabeled.parquet")
labeled_df = pd.read_parquet(config.PROCESSED_DIR / "labeled.parquet")

unlabeled_sample = stratified_sample(unlabeled_df, config.SAMPLE_SIZE, seed=config.SEED)

applier = PandasLFApplier(lfs=lfs)
L_train = applier.apply(df=unlabeled_sample)
L_dev = applier.apply(df=labeled_df)

print(LFAnalysis(L=L_train, lfs=lfs).lf_summary())
print(LFAnalysis(L=L_dev, lfs=lfs).lf_summary(Y=labeled_df["label"].to_numpy()))

  0%|          | 0/8000 [00:00<?, ?it/s]

 10%|█         | 819/8000 [00:00<00:00, 8129.96it/s]

 21%|██▏       | 1700/8000 [00:00<00:00, 8495.98it/s]

 32%|███▏      | 2598/8000 [00:00<00:00, 8713.99it/s]

 44%|████▎     | 3483/8000 [00:00<00:00, 8764.82it/s]

...

 74%|███████▍  | 4454/6000 [00:00<00:00, 8795.62it/s]

 89%|████████▉ | 5336/6000 [00:00<00:00, 8554.94it/s]

100%|██████████| 6000/6000 [00:00<00:00, 8505.94it/s]


C:\Users\ACER\OneDrive\Documents\final-project\.worktrees\autolabel-notebooks\.venv\Lib\site-packages\scipy\sparse\_construct.py:543: FutureWarning: Input has data type int64, but the output has been cast to float64.  In the future, the output data type will match the input. To avoid this warning, set the `dtype` parameter to `None` to have the output dtype match the input, or set it to the desired output data type.
Note: In Python 3.11, this warning can be generated by a call of scipy.sparse.diags(), but the code indicated in the warning message will refer to an internal call of scipy.sparse.diags_array(). If that happens, check your code for the use of diags().
  A = diags_array(diagonals, offsets=offsets, shape=shape, dtype=dtype)
C:\Users\ACER\OneDrive\Documents\final-project\.worktrees\autolabel-notebooks\.venv\Lib\site-packages\scipy\sparse\_construct.py:543: FutureWarning: Input has data type int64, but the output has been cast to float64.  In the future, the output data type w

                   j Polarity  Coverage  Overlaps  Conflicts
lf_world           0      [0]  0.250375  0.179625   0.179625
lf_sports          1      [1]  0.163875  0.102125   0.084250
lf_business        2      [2]  0.161875  0.114250   0.114250
lf_scitech         3      [3]  0.552750  0.296250   0.296250
lf_has_percentage  4       []  0.000000  0.000000   0.000000
lf_has_score       5      [1]  0.050875  0.039625   0.021750
                   j Polarity  Coverage  Overlaps  Conflicts  Correct  \
lf_world           0      [0]  0.258333  0.179000   0.179000      729   
lf_sports          1      [1]  0.165667  0.102167   0.086500      762   
lf_business        2      [2]  0.159333  0.111667   0.111667      646   
lf_scitech         3      [3]  0.555333  0.294333   0.294333      943   
lf_has_percentage  4       []  0.000000  0.000000   0.000000        0   
lf_has_score       5      [1]  0.051167  0.040667   0.025000      290   

                   Incorrect  Emp. Acc.  
lf_world           

In [4]:
overall_coverage = (L_train != ABSTAIN).any(axis=1).mean()
assert overall_coverage > 0.05, f"LF coverage suspiciously low: {overall_coverage:.2%}"
print(f"Overall LF coverage on unlabeled sample: {overall_coverage:.2%}")

Overall LF coverage on unlabeled sample: 78.84%


In [5]:
label_model = LabelModel(cardinality=config.NUM_CLASSES, verbose=True)
label_model.fit(L_train=L_train, n_epochs=500, lr=0.001, seed=config.SEED)

proba_labels = label_model.predict_proba(L=L_train)
hard_labels = label_model.predict(L=L_train)
confidence = proba_labels.max(axis=1)

INFO:root:Computing O...


INFO:root:Estimating \mu...


  0%|          | 0/500 [00:00<?, ?epoch/s]

INFO:root:[0 epochs]: TRAIN:[loss=0.360]


INFO:root:[10 epochs]: TRAIN:[loss=0.346]


...

INFO:root:[470 epochs]: TRAIN:[loss=0.003]


INFO:root:[480 epochs]: TRAIN:[loss=0.003]


INFO:root:[490 epochs]: TRAIN:[loss=0.003]


100%|██████████| 500/500 [00:00<00:00, 834.57epoch/s]


INFO:root:Finished Training


In [6]:
label_quality = evaluate_label_quality(
    true_labels=unlabeled_sample["true_label"].to_numpy(),
    pseudo_labels=hard_labels,
    confidence_scores=confidence)
print("Weak supervision label quality:", label_quality)

config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
with open(config.RESULTS_DIR / "metrics_weak_supervision.json", "w") as f:
    json.dump(label_quality, f, indent=2)

Weak supervision label quality: {'Label Accuracy': 0.46440462977643887, 'Label Macro F1': 0.4443517489675299, 'Coverage': np.float64(0.788375), 'Mean Confidence': 0.40828156497733886, 'Median Confidence': 0.3868349296214561}


### Plan B: if Snorkel won't install

Replace the `LabelModel` fit/predict cell above with this plain-pandas
weighted-vote combiner (less rigorous — no learned per-LF weights — but
dependency-free):

In [7]:
# import numpy as np
#
# def weighted_vote_labels(L):
#     votes = np.zeros((L.shape[0], config.NUM_CLASSES))
#     for col in range(L.shape[1]):
#         valid = L[:, col] != ABSTAIN
#         votes[valid, L[valid, col]] += 1
#     hard = votes.argmax(axis=1)
#     hard[votes.sum(axis=1) == 0] = ABSTAIN
#     return hard
#
# hard_labels = weighted_vote_labels(L_train)